## Block 1: Imports & Setup

In [1]:
import time
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report
)

from sklearn.calibration import CalibratedClassifierCV

RANDOM_STATE = 42

## Block 2: Load Dataset & Features

In [2]:
df = pd.read_csv("cleaned_dataset.csv")

raw_features = [
    "Log GDP per capita", "Social support", "Healthy life expectancy at birth",
    "Freedom to make life choices", "Generosity", "Perceptions of corruption",
    "Positive affect", "Negative affect"
]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[raw_features])
X = pd.DataFrame(X_scaled, columns=raw_features)
y = df["Happiness Level"]

## Block 3: Train/Validation/Test Split

In [3]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp)
#stratify=y → keeps the class distribution balanced across splits
#random_state=RANDOM_STATE → ensures reproducibility (same split every run).


## Block 4: Define Models

In [4]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "KNN": KNeighborsClassifier(n_neighbors=7),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=12, random_state=RANDOM_STATE),
    "SVM": CalibratedClassifierCV(
    estimator=SVC(kernel="rbf", random_state=RANDOM_STATE),
    ensemble=False   #radial basis function kernel
),
}

## Block 5: Train & Evaluate Models

In [5]:
y_test_bin = label_binarize(y_test, classes=[0, 1, 2, 3])

rows = []
fitted = {}
for name, model in models.items():
    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start

    val_pred = model.predict(X_val)
    val_acc = accuracy_score(y_val, val_pred)

    test_pred = model.predict(X_test)
    test_proba = model.predict_proba(X_test)

    acc = accuracy_score(y_test, test_pred)
    prec = precision_score(y_test, test_pred, average="macro", zero_division=0)
    rec = recall_score(y_test, test_pred, average="macro", zero_division=0)
    f1 = f1_score(y_test, test_pred, average="macro", zero_division=0)
    try:
        roc_auc = roc_auc_score(y_test_bin, test_proba, average="macro", multi_class="ovr")
    except ValueError:
        roc_auc = np.nan

    rows.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4),
        "ROC-AUC": round(roc_auc, 4),
        "Training Time (s)": round(train_time, 3),
        "Val Accuracy": round(val_acc, 4),
    })
    fitted[name] = model
    print(f"{name:22s} | test acc={acc:.4f}  val acc={val_acc:.4f}  train time={train_time:.3f}s")


Logistic Regression    | test acc=0.7380  val acc=0.7627  train time=0.033s
KNN                    | test acc=0.8000  val acc=0.8051  train time=0.007s
Decision Tree          | test acc=0.7577  val acc=0.7062  train time=0.019s
Random Forest          | test acc=0.8366  val acc=0.8136  train time=1.500s
SVM                    | test acc=0.7662  val acc=0.7797  train time=0.550s


## Block 6: Results & Best Model

In [6]:
results_df = pd.DataFrame(rows).sort_values("Val Accuracy", ascending=False).reset_index(drop=True)
print("\n", results_df.to_string(index=False))

best_name = results_df.iloc[0]["Model"]
best_model = fitted[best_name]
best_test_acc = results_df.iloc[0]["Accuracy"]
print(f"\nBest model (by validation accuracy): {best_name}")


               Model  Accuracy  Precision  Recall  F1-Score  ROC-AUC  Training Time (s)  Val Accuracy
      Random Forest    0.8366     0.8347  0.7240    0.7629   0.9544              1.500        0.8136
                KNN    0.8000     0.7408  0.6971    0.7133   0.9363              0.007        0.8051
                SVM    0.7662     0.7486  0.6325    0.6636   0.9242              0.550        0.7797
Logistic Regression    0.7380     0.6811  0.6098    0.6340   0.9199              0.033        0.7627
      Decision Tree    0.7577     0.7085  0.6704    0.6841   0.8640              0.019        0.7062

Best model (by validation accuracy): Random Forest


## Block 7: Classification Report

In [7]:
print("\nClassification report (test set, best model):")
print(classification_report(
    y_test, best_model.predict(X_test),
    target_names=["Unhappy", "Moderately Happy", "Happy", "Very Happy"]
))



Classification report (test set, best model):
                  precision    recall  f1-score   support

         Unhappy       0.83      0.45      0.59        33
Moderately Happy       0.85      0.95      0.90       202
           Happy       0.78      0.78      0.78        81
      Very Happy       0.88      0.72      0.79        39

        accuracy                           0.84       355
       macro avg       0.83      0.72      0.76       355
    weighted avg       0.84      0.84      0.83       355



## Block 8: Save Model & Results

In [8]:
label_map = {0: "Unhappy", 1: "Moderately Happy", 2: "Happy", 3: "Very Happy"}

# accuracy for every model, keyed by name (used by the model-select dropdown in the app)
test_accuracy_by_model = dict(zip(results_df["Model"], results_df["Accuracy"]))

bundle = {
    "models": fitted,                 # ALL trained models: name -> fitted estimator
    "best_model_name": best_name,      # e.g. "Random Forest"
    "scaler": scaler,
    "features": raw_features,
    "label_map": label_map,
    "countries": sorted(df["Country name"].unique().tolist()),
    "test_accuracy": test_accuracy_by_model,   # dict: name -> test accuracy
}
joblib.dump(bundle, "happyclassify_model.pkl")
results_df.drop(columns=["Val Accuracy"]).to_csv("model_comparison.csv", index=False)
print(f"\nSaved happyclassify_model.pkl (all {len(fitted)} models) and model_comparison.csv")



Saved happyclassify_model.pkl (all 5 models) and model_comparison.csv
